# TCC Rafael - Projeto 4

1. Raspagem e Estruturação de Dados de Touros Nelore CRV
2. Integração híbrida Loja Tray (preços de sêmen) + API Oficial CRV (réguas genéticas e biometria).

### Bibliotecas

In [45]:
import re
import json
import time
from typing import Dict, List, Optional, Any, Tuple
import requests
from bs4 import BeautifulSoup
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

### Configurações gerais

In [46]:
STORE_BASE_URL = "https://loja.crvbrasil.com.br/semen/touros-de-corte-zebu/nelore"
API_BASE_URL = "https://api-main-e6pbdwdwoa-rj.a.run.app/bull/find"

OUTPUT_CSV = "Dados/CSV/NeloreCRV.csv"
OUTPUT_EXCEL = "Dados/XLSX/NeloreCRV.xlsx"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Origin": "https://touros.crvbrasil.com.br",
    "Referer": "https://touros.crvbrasil.com.br/",
    "Accept": "application/json, text/plain, */*",
}

## Funções

### to_float

Converte valores textuais com vírgulas ou traço para float

In [47]:
def to_float(val: Any) -> Optional[float]:
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).strip().replace("R$", "").replace("%", "").strip()
    if s in ["-", "", "None", "null", "undefined"]:
        return None
    try:
        # Substitui vírgula decimal brasileira por ponto
        s_clean = s.replace(".", "").replace(",", ".") if s.count(",") == 1 and s.count(".") <= 1 else s.replace(",", ".")
        return float(s_clean)
    except Exception:
        try:
            return float(s.replace(",", "."))
        except Exception:
            return None

### collect_store_bulls
Varre as páginas do catálogo Nelore da loja CRV e extrai todos os touros à venda.

In [48]:
def collect_store_bulls(max_pages: int = 15) -> List[Dict[str, str]]:

    print(f"\n[FASE 1] Coletando touros da loja virtual: {STORE_BASE_URL}")
    bulls_store = []
    seen_urls = set()

    for pg in range(1, max_pages + 1):
        url = f"{STORE_BASE_URL}?pg={pg}"
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code != 200:
                print(f"  Página {pg}: status {r.status_code}. Encerrando paginação.")
                break

            soup = BeautifulSoup(r.text, "html.parser")
            cards = soup.find_all("li", class_=re.compile(r"item", re.IGNORECASE))
            new_in_page = 0

            for card in cards:
                name_el = card.find(class_=re.compile(r"product-name", re.IGNORECASE))
                link_el = card.find("a", href=re.compile(r"/nelore/[^?#]+"))
                price_el = card.find(class_=re.compile(r"price-off|PrecoPrincipal|product-price", re.IGNORECASE))

                if name_el and link_el:
                    href = link_el.get("href")
                    # Filtrar links que não são touros individuais
                    if href.endswith("/nelore") or href.endswith("/nelore-mocho"):
                        continue

                    if href not in seen_urls:
                        seen_urls.add(href)
                        bulls_store.append({
                            "store_name": name_el.get_text(strip=True),
                            "store_url": href,
                            "card_price_text": price_el.get_text(strip=True) if price_el else None
                        })
                        new_in_page += 1

            print(f"  Página {pg:2d}: {new_in_page:2d} novos touros encontrados. (Total: {len(bulls_store)})")
            if new_in_page == 0:
                print(f"[OK] Fim da paginação da loja na página {pg}.")
                break

            time.sleep(0.15)  # Throttling cortês
        except Exception as e:
            print(f"[ERRO] Falha ao ler página {pg}: {e}")
            break

    print(f"[FASE 1 CONCLUÍDA] Total de {len(bulls_store)} touros identificados na loja.\n")
    return bulls_store

### get_bull_prices_and_code

Visita a página do produto na loja e extrai o Código CRV e preços por dose.

In [49]:
def get_bull_prices_and_code(store_url: str) -> Tuple[Optional[str], Optional[float], Optional[float]]:
    try:
        r = requests.get(store_url, headers=HEADERS, timeout=20)
        if r.status_code != 200:
            return None, None, None
        text = r.text

        # 1. Extrair Código CRV
        crv_code = None
        code_match = re.search(r'"reference":\s*"([0-9A-Z]+)"', text)
        if code_match:
            crv_code = re.sub(r'CV$', '', code_match.group(1).strip())
        else:
            iframe_match = re.search(r'touros\.crvbrasil\.com\.br/[^"\']+/([0-9]{4,6})', text)
            if iframe_match:
                crv_code = iframe_match.group(1).strip()

        # 2. Preços (Convencional vs Sexada)
        price_conv = None
        price_sex = None

        # Tentar via dataLayer listSku
        sku_match = re.search(r'"listSku":\s*(\[.*?\])\s*,\s*"characteristcs"', text, re.DOTALL)
        if sku_match:
            try:
                skus = json.loads(sku_match.group(1))
                for sku in skus:
                    sku_name = sku.get("nameSku", "").lower()
                    p = sku.get("price") or sku.get("sellPrice")
                    if p:
                        p_val = float(p)
                        if "convencional" in sku_name:
                            price_conv = p_val
                        elif "sexad" in sku_name:
                            price_sex = p_val
                        elif not price_conv:
                            price_conv = p_val
            except Exception:
                pass

        # Fallback de preço se não encontrou nas SKUs
        if not price_conv:
            p_match = re.search(r'"priceSell":\s*([0-9\.]+)', text)
            if p_match:
                price_conv = float(p_match.group(1))

        return crv_code, price_conv, price_sex

    except Exception:
        return None, None, None


### fetch_bull_genetic_data 

Chama a API oficial da CRV e retorna a ficha zootécnica completa do touro.

In [50]:
# Criar uma sessão persistente com retentativas automáticas
session = requests.Session()
retries = Retry(
    total=3,
    backoff_factor=1.5,  # Espera 1.5s, depois 3s entre tentativas
    status_forcelist=[429, 500, 502, 503, 504],
    raise_on_status=False
)
session.mount("https://", HTTPAdapter(max_retries=retries))

def fetch_bull_genetic_data(crv_code: str, max_attempts: int = 3) -> Optional[Dict[str, Any]]:
    """
    Chama a API oficial da CRV com persistência de sessão e até 3 tentativas.
    """
    url = f"{API_BASE_URL}/{crv_code}"
    
    for attempt in range(1, max_attempts + 1):
        try:
            r = session.get(url, headers=HEADERS, timeout=45)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 404:
                # Touro realmente não cadastrado na API
                return None
            elif r.status_code == 429:
                # Rate limit temporário: espera um pouco mais
                time.sleep(2 * attempt)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            time.sleep(2 * attempt)
        except Exception:
            break
            
    return None

#### parse_pedigree

Extrai árvore genealógica (Pais e Avós).

In [51]:
def parse_pedigree(pedigree_list: List[Dict[str, Any]]) -> Dict[str, Optional[str]]:
    
    ped: Dict[str, Optional[str]] = {
        "pai": None, "pai_reg": None,
        "mae": None, "mae_reg": None,
        "avo_paterno": None, "avo_materno": None,
    }
    if not pedigree_list or not isinstance(pedigree_list, list):
        return ped

    first = pedigree_list[0]
    pai = first.get("Pai") or {}
    mae = first.get("Mae") or {}

    ped["pai"] = pai.get("name")
    ped["pai_reg"] = pai.get("code")
    ped["mae"] = mae.get("name")
    ped["mae_reg"] = mae.get("code")

    if pai.get("pai"):
        ped["avo_paterno"] = pai.get("pai", {}).get("name")
    if mae.get("pai"):
        ped["avo_materno"] = mae.get("pai", {}).get("name")

    return ped

### parse_proof_evaluations

Varre dinamicamente todos os sumários genéticos (ANCP, PMGZ, GENEPLUS, etc.) e extrai DEPs, Acurácias e Percentis TOP / Decas.

In [52]:
def parse_proof_evaluations(proof_list: List[Dict[str, Any]]) -> Dict[str, Any]:
    
    eval_data: Dict[str, Any] = {}
    if not proof_list or not isinstance(proof_list, list):
        return eval_data

    for item in proof_list:
        if not isinstance(item, dict):
            continue
        for summary_name, summary_content in item.items():
            if summary_name == "CRVcode" or not isinstance(summary_content, dict):
                continue

            # Identificar prefixo padrão (ANCP, PMGZ, GENEPLUS, PAINT, etc.)
            prefix = summary_name.split()[0].upper()
            if "ANCP" in summary_name.upper():
                prefix = "ANCP"
            elif "PMGZ" in summary_name.upper():
                prefix = "PMGZ"
            elif "GENEPLUS" in summary_name.upper():
                prefix = "GENEPLUS"

            tables = summary_content.get("table", [])
            for tbl in tables:
                headers = tbl.get("headers", [])
                types = tbl.get("type", [])
                values = tbl.get("values", [])

                # Se types estiver vazio (ex: Progênie), pula
                if not types or not values:
                    continue

                for i, trait_name in enumerate(headers):
                    if i < len(values):
                        row_vals = values[i]
                        for j, metric_type in enumerate(types):
                            if j < len(row_vals):
                                raw_val = row_vals[j]
                                clean_val = to_float(raw_val)

                                # Padronizar nome do tipo de métrica
                                mt_clean = metric_type.replace("%", "").replace("°", "").strip().upper()
                                col_name = f"{prefix}_{trait_name}_{mt_clean}"

                                eval_data[col_name] = clean_val

    return eval_data

### run_scraper

Integra as funções anteriores e realiza o scraper dos dados da plataforma CRV

In [53]:
def run_scraper():
    start_time = time.time()
    print("=" * 75)
    print("INICIANDO SCRAPER E MODELAGEM DE DADOS - CRV NELORE")
    print("=" * 75)

    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    # Fase 01. Varredura da vitrina da loja (preço e URLs)
    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    store_bulls = collect_store_bulls(max_pages=15)
    if not store_bulls:
        print("[ERRO] Nenhum touro encontrado na loja. Encerrando.")
        return

    print(f"Processando {len(store_bulls)} touros (Preços + API Oficial CRV)...")
    consolidated_records: List[Dict[str, Any]] = []

    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    # Fase 02: Detalhes de preço por dose (convencional e sexada) e código CRV
    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    for idx, bull in enumerate(store_bulls, start=1):
        name = bull["store_name"]
        url = bull["store_url"]

        # Obter Preço e Código CRV da página da loja
        crv_code, price_conv, price_sex = get_bull_prices_and_code(url)

        # Registro base do touro
        record: Dict[str, Any] = {
            "crv_code": crv_code,
            "nome_loja": name,
            "preco_convencional": price_conv,
            "preco_sexado": price_sex,
            "url_loja": url,
        }

        # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
        # Fase 03: Consulta à API oficial CRV e extração de réguas genéticas
        # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
        if crv_code: # Se tivermos o código CRV, consultar a API oficial
            api_data = fetch_bull_genetic_data(crv_code)
            if api_data:
                # 1. Identificação Oficial
                record["nome_oficial"] = api_data.get("name")
                record["nome_guerra"] = api_data.get("warName")
                record["raca"] = api_data.get("breed")
                record["rgd"] = api_data.get("registry")
                record["data_nascimento"] = api_data.get("birthDate")
                record["linhagem"] = api_data.get("lineage")
                record["criador"] = api_data.get("creator")
                record["proprietario"] = api_data.get("owner")
                record["sistema_criacao"] = api_data.get("system")
                record["idade_medicao"] = api_data.get("ageInMeasure")

                # 2. Biometria Corporal
                record["peso_kg"] = to_float(api_data.get("weight"))
                record["circunferencia_escrotal_cm"] = to_float(api_data.get("CE"))
                record["comprimento_corporal_cm"] = to_float(api_data.get("bodyLength"))
                record["altura_garupa_cm"] = to_float(api_data.get("rumpHeight"))
                record["largura_garupa_cm"] = to_float(api_data.get("rumpWidth"))
                record["comprimento_garupa_cm"] = to_float(api_data.get("croupLength"))
                record["profundidade_torax_cm"] = to_float(api_data.get("chestDepth"))
                record["perimetro_toracico_cm"] = to_float(api_data.get("thoracicPerimeter"))

                # 3. Selos Comerciais
                stamps = api_data.get("stamps", [])
                record["selos"] = "; ".join(stamps) if isinstance(stamps, list) else None

                # 4. Pedigree
                ped = parse_pedigree(api_data.get("pedigree", []))
                record.update(ped)

                # 5. Avaliações Genéticas e Réguas (ANCP, PMGZ, GENEPLUS)
                eval_metrics = parse_proof_evaluations(api_data.get("proof", []))
                record.update(eval_metrics)

                print(f"[{idx:3d}/{len(store_bulls)}] {name[:30]:<30} | CRV: {crv_code} | Preço: R${price_conv or 0:5.2f} | Réguas: {len(eval_metrics)} vars")
            else:
                print(f"[{idx:3d}/{len(store_bulls)}] {name[:30]:<30} | CRV: {crv_code} | [AVISO: Sem retorno na API]")
        else:
            print(f"[{idx:3d}/{len(store_bulls)}] {name[:30]:<30} | [AVISO: Código CRV não detectado]")

        consolidated_records.append(record)
        time.sleep(0.4)  # Intervalo de segurança para estabilidade

    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    # Fase 04: Exportação dos dados
    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    df = pd.DataFrame(consolidated_records)

    # Organização das colunas principais na frente
    first_cols = [
        "crv_code", "nome_oficial", "nome_guerra", "raca", "rgd", "data_nascimento",
        "preco_convencional", "preco_sexado",
        "ANCP_MGTe_DEP", "ANCP_MGTe_TOP",
        "PMGZ_iABCZ_DEP", "PMGZ_iABCZ_DECA",
        "GENEPLUS_IQG_DEP", "GENEPLUS_IQG_TOP",
        "peso_kg", "circunferencia_escrotal_cm", "altura_garupa_cm",
        "criador", "proprietario", "linhagem", "pai", "mae", "selos", "url_loja"
    ]
    existing_first = [c for c in first_cols if c in df.columns]
    other_cols = sorted([c for c in df.columns if c not in existing_first])
    final_cols = existing_first + other_cols
    df = df[final_cols]

    # Salvar em CSV e Excel
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    df.to_excel(OUTPUT_EXCEL, index=False)

    total_time = time.time() - start_time
    print("\n" + "=" * 75)
    print(f"[SUCESSO] Pipeline finalizado em {total_time:.1f} segundos ({total_time/60:.2f} min)!")
    print(f"Total de registros exportados: {len(df)}")
    print(f"Total de colunas (variáveis): {len(df.columns)}")
    print(f"Arquivos salvos:")
    print(f"  -> CSV  : {OUTPUT_CSV}")
    print(f"  -> Excel: {OUTPUT_EXCEL}")
    print("=" * 75)

### main

Chama a função run_scraper

In [54]:
if __name__ == "__main__":
    run_scraper()

INICIANDO SCRAPER E MODELAGEM DE DADOS - CRV NELORE

[FASE 1] Coletando touros da loja virtual: https://loja.crvbrasil.com.br/semen/touros-de-corte-zebu/nelore
  Página  1: 48 novos touros encontrados. (Total: 48)
  Página  2: 48 novos touros encontrados. (Total: 96)
  Página  3: 47 novos touros encontrados. (Total: 143)
  Página  4: 48 novos touros encontrados. (Total: 191)
  Página  5: 48 novos touros encontrados. (Total: 239)
  Página  6: 48 novos touros encontrados. (Total: 287)
  Página  7: 48 novos touros encontrados. (Total: 335)
  Página  8:  3 novos touros encontrados. (Total: 338)
  Página  9:  0 novos touros encontrados. (Total: 338)
[OK] Fim da paginação da loja na página 9.
[FASE 1 CONCLUÍDA] Total de 338 touros identificados na loja.

Processando 338 touros (Preços + API Oficial CRV)...
[  1/338] ACE CACHOEIRÃO - NELORE        | CRV: 09006 | Preço: R$33.90 | Réguas: 158 vars
[  2/338] ALIADO DA BAMA - NELORE        | CRV: 09031 | Preço: R$31.20 | Réguas: 158 vars
[  3/338